In [ ]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ (and its bowaka_common dependency) to sys.path and pins
# the working directory to the repo root, so `import bowaka_v2_lab` and
# repo-root-relative CONFIG_PATH parameters resolve identically under jupyter,
# papermill, and the QuantsLab scheduler.
import os
import sys
from pathlib import Path

_lab_root = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        _lab_root = _candidate
        break
if _lab_root is None:
    raise RuntimeError(
        f"bowaka_v2_lab bootstrap: src/bowaka_v2_lab/ not found at or above {Path.cwd()}"
    )

# Pin CWD to the repo root so repo-root-relative CONFIG_PATH values resolve
# regardless of how the notebook was launched (jupyter CWD = notebook dir,
# scheduler = repo root). The lab ALWAYS lives at
# ``<repo_root>/research_notebooks/bowaka_v2_lab``, so derive the repo root from
# that canonical layout. A marker-only heuristic ("a dir with research_notebooks/
# AND Makefile") mis-matches the lab dir itself when it carries a Makefile and a
# stray nested research_notebooks/ — which then chdir's one level too deep and
# breaks every repo-root-relative path.
if _lab_root.parent.name == "research_notebooks":
    _repo_root = _lab_root.parent.parent
else:
    # Fallback: the repo root holds research_notebooks/bowaka_common (the sibling
    # package) — a marker a stray nested research_notebooks/ inside the lab lacks.
    _repo_root = _lab_root
    for _candidate in [_lab_root, *_lab_root.parents]:
        if (_candidate / "research_notebooks" / "bowaka_common").is_dir():
            _repo_root = _candidate
            break
os.chdir(_repo_root)

# Make the lab and its bowaka_common dependency importable from the working
# tree, even when the packages are not pip-installed. v1 bowaka_lab is
# deliberately excluded — v2 must not import v1.
for _src in (_lab_root / "src",
             _repo_root / "research_notebooks" / "bowaka_common" / "src"):
    if _src.is_dir() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))

import bowaka_v2_lab  # noqa: F401
print(f"bowaka_v2_lab {bowaka_v2_lab.__version__} (cwd={_repo_root})")


In [ ]:
# Papermill parameter cell.
CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_backtest_smoke.yml'


# 01 — Shared Data Inventory

Inventories local `data/fixtures/` **and** the shared market-data lake.

In [ ]:
from pathlib import Path
from bowaka_v2_lab.config import load_config, BowakaV2Paths
from bowaka_v2_lab.config.models import BowakaV2Config
from bowaka_common.marketdata import available_symbols, date_coverage, resolve_market_data_root
cfg = load_config(CONFIG_PATH)
validated = BowakaV2Config.model_validate(cfg)
paths = BowakaV2Paths.from_config(validated, repo_root=Path('.').resolve())
_feed = cfg.get('market_data', {}).get('feed', 'iex')
# --- local fixtures ---
fix_root = Path(paths.data_root) / 'fixtures'
files = sorted(p.relative_to(fix_root) for p in fix_root.rglob('*.parquet')) if fix_root.is_dir() else []
print(f'local fixtures: {len(files)} parquet file(s) under {fix_root}')
for f in files[:20]:
    print('  ', f)
# --- shared market-data lake ---
lake_root = resolve_market_data_root(cfg.get('market_data', {}).get('shared_root'), create=False)
daily_syms = available_symbols(lake_root, timeframe='1d', feed=_feed)
minute_syms = available_symbols(lake_root, timeframe='1m', feed=_feed)
print(f'shared lake: {lake_root} (feed={_feed})')
print(f'  daily-bar symbols:  {len(daily_syms)}')
print(f'  minute-bar symbols: {len(minute_syms)}')
for sym in daily_syms[:10]:
    cov = date_coverage(sym, lake_root, timeframe='1d', feed=_feed)
    print(f'    {sym}: ' + (f'{cov[0]} -> {cov[1]}' if cov else '(no coverage)'))
